# 🦙 Ollama — Test de conexión desde JupyterHub

Este notebook verifica la conectividad con Ollama desde el overlay interno de Swarm.

| Contexto | URL correcta | Auth |
|----------|-------------|------|
| **JupyterHub** (dentro del cluster) | `http://ollama:11434` | ❌ no requerida |
| **Postman / equipo local** (LAN `192.168.80.0/24`) | `http://192.168.80.200:11434` | ❌ no tiene middleware |
| **Externo vía Traefik** | `https://ollama.sexydad` | ✅ BasicAuth requerida |

> **Error frecuente:** usar `http://192.168.80.200:11434` desde JupyterHub provoca  
> `ConnectTimeout` porque la IP del host no es alcanzable desde el network namespace  
> del contenedor Swarm. La ruta correcta es siempre el DNS del overlay: `ollama`.

## 1. Cargar configuración desde `apis.env`

El archivo vive en `~/work/.config/llm/apis.env` (no sube al repo, está en `.gitignore`).
Copiá el template desde `notebooks/config/apis.env.template` si aún no existe.

In [ ]:
from pathlib import Path
import os
import requests
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

env_path = Path.home() / "work" / ".config" / "llm" / "apis.env"

# override=False: si JupyterHub ya inyectó OLLAMA_BASE_URL en el entorno
# (vía SwarmSpawner.environment en jupyterhub_config.py), esa variable tiene
# prioridad y el archivo NO la sobreescribe.
load_dotenv(env_path, override=False)

if not env_path.exists():
    print(f"⚠️  Archivo no encontrado: {env_path}")
    print("   Copiá el template: notebooks/config/apis.env.template → ~/work/.config/llm/apis.env")

ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://ollama:11434").rstrip("/")

# Auth solo aplica para el endpoint externo (https://ollama.sexydad).
# En la red interna del overlay, Ollama no tiene middleware de autenticación.
_username = os.environ.get("OLLAMA_USERNAME")
_password = os.environ.get("OLLAMA_PASSWORD")
auth = HTTPBasicAuth(_username, _password) if (_username and _password) else None

print(f"OLLAMA_BASE_URL : {ollama_base_url}")
print(f"Auth habilitada : {'sí' if auth else 'no (modo interno)'}")

## 2. Verificar DNS del overlay

In [ ]:
import socket

hostname = "ollama"
try:
    ip = socket.gethostbyname(hostname)
    print(f"✅ DNS overlay resuelto: {hostname} → {ip}")
except socket.gaierror as exc:
    print(f"❌ No se resuelve '{hostname}': {exc}")
    print("   Verificá que el servicio ollama_ollama está corriendo y el container está en la red 'internal'.")

## 3. Health check — listar modelos

In [ ]:
try:
    response = requests.get(
        f"{ollama_base_url}/api/tags",
        auth=auth,
        timeout=15,
    )
    print(f"HTTP {response.status_code}")
    if response.ok:
        models = response.json().get("models", [])
        if models:
            print(f"✅ {len(models)} modelo(s) disponible(s):")
            for m in models:
                size_gb = m.get("size", 0) / 1e9
                print(f"   • {m['name']:40s}  {size_gb:.1f} GB")
        else:
            print("⚠️  Conexión ok pero sin modelos descargados.")
            print("   Para descargar: docker exec $(docker ps -q -f name=ollama_ollama) ollama pull llama3.2:3b")
    else:
        print(f"❌ Error HTTP: {response.text[:300]}")
except requests.exceptions.ConnectTimeout:
    print("❌ ConnectTimeout — posibles causas:")
    print(f"   1. OLLAMA_BASE_URL apunta al host ({ollama_base_url}) en vez del overlay.")
    print("      Solución: cambiar a http://ollama:11434 en ~/work/.config/llm/apis.env")
    print("   2. El servicio ollama_ollama no está corriendo.")
    print("      Verificar: docker service ls | grep ollama  (desde master1)")
except requests.RequestException as exc:
    print(f"❌ Error de conexión: {type(exc).__name__}: {exc}")

## 4. Inferencia de prueba

In [ ]:
import json

MODEL = "llama3.2:3b"   # cambiar si usás otro modelo
PROMPT = "En una oración: ¿qué es un Docker overlay network?"

try:
    r = requests.post(
        f"{ollama_base_url}/api/generate",
        auth=auth,
        json={"model": MODEL, "prompt": PROMPT, "stream": False},
        timeout=120,
    )
    r.raise_for_status()
    print(f"Modelo : {MODEL}")
    print(f"Prompt : {PROMPT}")
    print(f"\nRespuesta:\n{r.json()['response']}")
except requests.RequestException as exc:
    print(f"❌ {type(exc).__name__}: {exc}")

## 5. Fix rápido — actualizar `apis.env` desde el notebook

Si el archivo tiene la IP del host, ejecutá esta celda para corregirlo.

In [ ]:
import re

CORRECT_URL = "http://ollama:11434"

if not env_path.exists():
    print(f"Archivo no encontrado: {env_path}")
else:
    content = env_path.read_text()
    # Reemplaza cualquier valor de OLLAMA_BASE_URL por la URL interna
    updated = re.sub(
        r"^(OLLAMA_BASE_URL\s*=\s*).*$",
        rf"\g<1>{CORRECT_URL}",
        content,
        flags=re.MULTILINE,
    )
    if updated == content:
        current = re.search(r"^OLLAMA_BASE_URL\s*=\s*(.+)$", content, re.MULTILINE)
        val = current.group(1).strip() if current else "(no encontrada)"
        print(f"Sin cambios — valor actual: {val}")
    else:
        env_path.write_text(updated)
        print(f"✅ OLLAMA_BASE_URL actualizada a: {CORRECT_URL}")
        print("   Reiniciá el kernel para que el cambio tome efecto.")